# 04 — Churn Risk Modeling

**Business question:** Which customers are most likely to churn?

Start with logistic regression for interpretability, then optionally compare XGBoost/LightGBM.
Probability calibration matters because later financial decisions depend on probabilities.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.features import add_business_features
from src.models import build_logistic_model, evaluate_probabilistic_classifier

path = ROOT/"data/processed/demo_customers.csv"
if not path.exists():
    exec((ROOT/"data/generate_demo_data.py").read_text())
    main()

df = add_business_features(pd.read_csv(path))
model_df = df.drop(columns=["customer_id"], errors="ignore").copy()

train, test = train_test_split(
    model_df, test_size=0.25, random_state=42, stratify=model_df["churn_flag"]
)

X_train = train.drop(columns=["churn_flag"])
y_train = train["churn_flag"]
X_test = test.drop(columns=["churn_flag"])
y_test = test["churn_flag"]

train_for_builder = train.copy()
model = build_logistic_model(train_for_builder)
model.fit(X_train, y_train)

metrics = evaluate_probabilistic_classifier(model, X_test, y_test)
metrics

In [ ]:
test_scored = test.copy()
test_scored["churn_probability"] = model.predict_proba(X_test)[:,1]
test_scored["risk_segment_ml"] = pd.cut(
    test_scored["churn_probability"],
    bins=[0,0.30,0.60,1.0],
    labels=["Low","Medium","High"],
    include_lowest=True
)
test_scored[["churn_probability","risk_segment_ml","churn_flag"]].head()

### Optional extension

Use `src.models.fit_xgboost()` and `fit_lightgbm()` to compare boosted-tree models.
Keep logistic regression as the business-interpretable baseline.